# Lesson 13 Lab — Coalescing, Strides, and Shared-Memory Staging

**Puzzle:** Two tensors contain the same number of values; why can copying a transposed view be slower than copying a contiguous tensor?

This notebook retains one complete RTX 5090 execution.


## Why this matters

Global memory requests from a warp are combined into transactions according to the address segments touched. Adjacent lanes accessing adjacent words tend to use transferred bytes efficiently; strided patterns may require more transactions for the same useful bytes. Shared memory can stage a tile and change access order, but it adds loads, stores, synchronization, capacity use, and possible bank conflicts.


## 0. Predict before running

1. Predict the strides of a tensor and its transpose.
2. Predict which copy has higher requested bandwidth.
3. List the costs introduced by a shared-memory transpose tile.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The lab copies a contiguous 2-D CUDA tensor and its non-contiguous transpose into fresh contiguous outputs. Both expose the same logical element count and dtype, so requested bytes match. CUDA-event time and effective bandwidth show the layout consequence through PyTorch's copy kernels. A direct hardware-transaction claim still requires global-load/store sector counters.

- Coalescing is evaluated across addresses requested by a warp instruction.
- A view can change strides without changing logical shape or storage ownership.
- Shared-memory tiling is useful when it converts repeated or strided global access into reused, organized access.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["warp addresses"] --> B["32-byte segments"]
  B --> C["global transactions"]
  C --> D["shared-memory tile"]
  D --> E["reordered/reused access"]
  E --> F["coalesced output"]
```


## 3. Inspect the visual boundary

This lesson is driven by a Mermaid mechanism map and executable measurements.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 13
LESSON_TITLE = 'Coalescing, Strides, and Shared-Memory Staging'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260826
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | contiguous source to contiguous destination |
| Candidate | transposed non-contiguous view to contiguous destination |
| Held constant | logical elements, dtype, destination layout, warm-up, and event timing |
| Measurements | source strides, median latency, effective GB/s, and slowdown |
| Evidence | `pytorch-gpu` |

**Experiment:** Copy equal-sized contiguous and transposed CUDA views.


## 6. Inspect the code

Preallocated outputs avoid allocator timing. `copy_` consumes either the base tensor or its transpose, and checksums establish equivalent content after accounting for transpose order.

Do not run until the code matches the frozen table.


In [2]:
n = 8192
base = torch.randn((n, n), device=DEVICE, dtype=torch.float32)
transposed = base.t()
out_contiguous = torch.empty_like(base)
out_transposed = torch.empty_like(base)
contiguous_samples = cuda_samples(lambda: out_contiguous.copy_(base), repeats=20)
transposed_samples = cuda_samples(lambda: out_transposed.copy_(transposed), repeats=20)
contiguous_median = statistics.median(contiguous_samples)
transposed_median = statistics.median(transposed_samples)
requested_bytes = 2 * base.numel() * base.element_size()
contiguous_gbps = requested_bytes / (contiguous_median / 1e3) / 1e9
transposed_gbps = requested_bytes / (transposed_median / 1e3) / 1e9
equivalent = bool(torch.allclose(out_transposed[:64, :64], base.t()[:64, :64]))
metrics = {
    "shape": list(base.shape), "base_stride": list(base.stride()),
    "transposed_stride": list(transposed.stride()),
    "contiguous_median_ms": contiguous_median,
    "transposed_median_ms": transposed_median,
    "contiguous_gbps": contiguous_gbps, "transposed_gbps": transposed_gbps,
    "transposed_slowdown": transposed_median / contiguous_median,
    "output_equivalent": equivalent,
    "contiguous_samples_ms": contiguous_samples, "transposed_samples_ms": transposed_samples,
}
analysis = (
    f"The contiguous and transposed views requested the same logical bytes, but their strides "
    f"were {base.stride()} and {transposed.stride()}; copy latency changed by "
    f"{metrics['transposed_slowdown']:.3f}x. Physical transaction counters were not collected."
)
print(json.dumps(metrics, indent=2))


{
  "shape": [
    8192,
    8192
  ],
  "base_stride": [
    8192,
    1
  ],
  "transposed_stride": [
    1,
    8192
  ],
  "contiguous_median_ms": 0.35391999781131744,
  "transposed_median_ms": 0.6519840061664581,
  "contiguous_gbps": 1516.927314986642,
  "transposed_gbps": 823.4418435456703,
  "transposed_slowdown": 1.8421790523236983,
  "output_equivalent": true,
  "contiguous_samples_ms": [
    0.35628798604011536,
    0.3569599986076355,
    0.35411199927330017,
    0.35468798875808716,
    0.3537279963493347,
    0.3547520041465759,
    0.35446399450302124,
    0.3521279990673065,
    0.3542720079421997,
    0.35519999265670776,
    0.355103999376297,
    0.3534719944000244,
    0.3572799861431122,
    0.3532800078392029,
    0.3534719944000244,
    0.35369598865509033,
    0.35369598865509033,
    0.3529919981956482,
    0.3537279963493347,
    0.35343998670578003
  ],
  "transposed_samples_ms": [
    0.669439971446991,
    0.6566720008850098,
    0.6520320177078247,
    0.65

## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Contiguous median | 0.354 ms |
| Transposed median | 0.652 ms |
| Contiguous bandwidth | 1,516.9273 |
| Transposed bandwidth | 823.4418 |
| Transposed slowdown | 1.842x |


## 8. Explain rather than overclaim

The contiguous and transposed views requested the same logical bytes, but their strides were (8192, 1) and (1, 8192); copy latency changed by 1.842x. Physical transaction counters were not collected.

**Evidence boundary:** CUDA work executed through PyTorch. It does not identify an internal instruction, cache event, or proprietary hardware block without additional profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 13, "title": 'Coalescing, Strides, and Shared-Memory Staging', "environment": ENV,
    "evidence_label": 'pytorch-gpu', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Fix global access order before adding arithmetic micro-optimizations; introduce shared memory only with an explicit reuse or reordering purpose.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 13,
  "title": "Coalescing, Strides, and Shared-Memory Staging",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260826
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "shape": [
      8192,
      8192
    ],
    "base_stride": [
      8192,
      1
    ],
    "transposed_stride": [
      1,
      8192
    ],
    "contiguous_median_ms": 0.35391999781131744,
    "transposed_median_ms": 0.6519840061664581,
    "contiguous_gbps": 1516.927314986642,
    "transposed_gbps": 823.4418435456703,
    "transposed_slowdown": 1.8421790523236983,
    "output_equivalent": true,
    "contiguous_samples_ms": [
      0.35628798604011536,
      0.3569599986076355,
      0.35411199927330017,
      0.35468798875808716,
      0.3537279963493347,
      0.3547520041465759,
      0.35446399450302124,
      0.3521279990673065,
      0.3542720079421997

## 10. Make the decision

> Fix global access order before adding arithmetic micro-optimizations; introduce shared memory only with an explicit reuse or reordering purpose.

**Failure analysis:** PyTorch may use specialized copy kernels, and cache state plus matrix dimensions affect results. Requested bandwidth does not equal physical bus traffic.


## 11. Extend the evidence

Write naive and tiled CUDA transpose kernels and compare global sectors, shared bank conflicts, and end-to-end bandwidth.

See [`README.md`](README.md) for the full explanation and references.
